# 12. Gierige Verfahren: immer die scheinbar beste Entscheidung

In [Notebook 11](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/11_tabellen_dp.ipynb) hast du Tabellen gefüllt und dabei **alle** Möglichkeiten für einen Eintrag
verglichen. Bei einem **gierigen Verfahren** (englisch *greedy*) spart man sich das: Man trifft in jedem Schritt die Entscheidung, die
**im Moment** am besten aussieht, und kommt nie darauf zurück. Das ist meistens sehr schnell und einfach zu programmieren. Der Haken:
Es ist **nicht immer richtig**. Bei den Münzen aus Notebook 10 mit `MUENZEN = (1, 4, 6)` zahlt man 8 mit 4 + 4 (zwei Münzen), wer aber
gierig immer die größte Münze nimmt, bekommt 6 + 1 + 1 (drei Münzen).

Deshalb gehört zu jedem gierigen Verfahren in der 1. Runde eine **Begründung**, warum es stimmt, oder ein Vergleich mit einer genauen
Lösung. In diesem Notebook lernst du:

- gierige Verfahren zu schreiben: **sortieren** nach einem Schlüssel (`sorted` mit `key=lambda`, kurze Wiederholung aus Notebook 3) und dann
  der Reihe nach entscheiden,
- **Gegenbeispiele** zu finden, indem man das gierige Verfahren mit **Durchprobieren** (Notebook 1) vergleicht, auch mit Zufallsdaten,
- ein **Austauschargument** zu führen: Warum darf man die gierige Wahl nehmen, ohne etwas zu verlieren?
- aus mehreren gierigen Regeln („kürzester zuerst“, „früh endend zuerst“) die richtige zu wählen,
- zu erkennen, dass ein erfolgreicher Test **nichts beweist** (aber ein Gegenbeispiel alles widerlegt).

In der Wettbewerbs-Ecke planst du Tankstopps für eine Urlaubsfahrt, und für Schnelle ordnest du beim Frühsport Ringel mit möglichst
wenigen Tauschschritten.

## 1. Wechselgeld: wann ist „immer die größte Münze“ richtig?

Die Aufgabe aus Notebook 10 und 11: Man soll einen Betrag mit möglichst **wenigen** Münzen zahlen. Das gierige Verfahren:
**Nimm immer die größte Münze, die noch in den Betrag passt**, und wiederhole das mit dem Rest.

Um die Münzen nach der Größe zu ordnen, benutzt man `sorted(muenzen, reverse=True)` (absteigend). Das kennst du schon: Zum Sortieren
nach einem anderen Schlüssel als dem Wert selbst gibt es `key=lambda`, zum Beispiel `sorted(dinge, key=lambda d: d[2])` (Notebook 3). Das
brauchst du gleich bei den Terminen.

#### Aufgabe 1.1

Schreibe `gierig_muenzen(betrag, muenzen)`: Sie gibt die Liste der Münzen zurück, die das gierige Verfahren wählt: Gehe die Münzen von der
größten zur kleinsten durch und nimm jede so oft, wie sie in den Rest passt. Ein Betrag, der nicht aufgeht, kommt hier nicht vor
(wir haben immer die Münze 1).

**Erwartete Ausgabe:**
```
8 -> [6, 1, 1] , 3 Münzen
12 -> [6, 6] , 2 Münzen
20 -> [6, 6, 6, 1, 1] , 5 Münzen
25 -> [6, 6, 6, 6, 1] , 5 Münzen
[50, 20, 20, 1]
```

In [ ]:
def gierig_muenzen(betrag, muenzen):
    plan = []
    # dein Code
    return plan

for betrag in [8, 12, 20, 25]:
    plan = gierig_muenzen(betrag, (1, 4, 6))
    print(betrag, '->', plan, ',', len(plan), 'Münzen')
print(gierig_muenzen(91, (1, 2, 5, 10, 20, 50)))

<details>
<summary>Tipp anzeigen</summary>

`for muenze in sorted(muenzen, reverse=True):` und darin `while betrag >= muenze:` die Münze an `plan` anhängen und vom Betrag abziehen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def gierig_muenzen(betrag, muenzen):
    plan = []
    for muenze in sorted(muenzen, reverse=True):     # von der größten zur kleinsten
        while betrag >= muenze:                      # so oft nehmen, wie sie noch passt
            plan.append(muenze)
            betrag = betrag - muenze
    return plan

for betrag in [8, 12, 20, 25]:
    plan = gierig_muenzen(betrag, (1, 4, 6))
    print(betrag, '->', plan, ',', len(plan), 'Münzen')
print(gierig_muenzen(91, (1, 2, 5, 10, 20, 50)))
```

Mit den Münzen 1, 4 und 6 gibt das Verfahren für 8 die Münzen `[6, 1, 1]` (3 Stück), für 20 `[6, 6, 6, 1, 1]` (5 Stück); bei 12 (`[6, 6]`) und 25 (`[6, 6, 6, 6, 1]`)
ist es richtig. Die besten Pläne aus Notebook 10 haben bei 8 und 20 nur 2 und 4 Münzen (`[4, 4]` und `[4, 4, 6, 6]`). Bei den Euro-Münzen
passt der Plan für 91: `50 + 20 + 20 + 1`.

</details>

#### Aufgabe 1.2

So findest du ein **Gegenbeispiel**: Vergleiche das gierige Verfahren mit der genauen Lösung. Die Funktion `wenigste_tabelle` aus Notebook 11 ist
unten schon fertig. Schreibe `schlechte_betraege(muenzen, maximum)`: Sie gibt die Liste der Beträge von 1 bis `maximum` zurück, bei denen das
gierige Verfahren **mehr** Münzen braucht als nötig.

**Erwartete Ausgabe:**
```
(1, 4, 6) : 65 von 200 Beträgen falsch, die ersten: [8, 9, 14, 15, 20, 21]
(1, 3, 4) : 49 von 200 Beträgen falsch, die ersten: [6, 10, 14, 18, 22, 26]
(1, 2, 5, 10, 20, 50) : 0 von 200 Beträgen falsch, die ersten: []
(1, 5, 10, 25) : 0 von 200 Beträgen falsch, die ersten: []
```

In [ ]:
def gierig_muenzen(betrag, muenzen):
    plan = []
    for muenze in sorted(muenzen, reverse=True):
        while betrag >= muenze:
            plan.append(muenze)
            betrag = betrag - muenze
    return plan

def wenigste_tabelle(muenzen, maximum):
    """Wie in Notebook 11: tabelle[b] ist die wenigste Münzzahl für den Betrag b."""
    tabelle = [None] * (maximum + 1)
    tabelle[0] = 0
    for betrag in range(1, maximum + 1):
        for muenze in muenzen:
            if muenze <= betrag and tabelle[betrag - muenze] is not None:
                if tabelle[betrag] is None or tabelle[betrag - muenze] + 1 < tabelle[betrag]:
                    tabelle[betrag] = tabelle[betrag - muenze] + 1
    return tabelle

def schlechte_betraege(muenzen, maximum):
    tabelle = wenigste_tabelle(muenzen, maximum)
    schlecht = []
    # dein Code
    return schlecht

for muenzen in [(1, 4, 6), (1, 3, 4), (1, 2, 5, 10, 20, 50), (1, 5, 10, 25)]:
    schlecht = schlechte_betraege(muenzen, 200)
    print(muenzen, ':', len(schlecht), 'von 200 Beträgen falsch, die ersten:', schlecht[:6])

<details>
<summary>Tipp anzeigen</summary>

Für jeden `betrag` von 1 bis `maximum`: Vergleiche `len(gierig_muenzen(betrag, muenzen))` mit `tabelle[betrag]`. Sind sie verschieden,
hänge `betrag` an `schlecht` an.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def gierig_muenzen(betrag, muenzen):
    plan = []
    for muenze in sorted(muenzen, reverse=True):
        while betrag >= muenze:
            plan.append(muenze)
            betrag = betrag - muenze
    return plan

def wenigste_tabelle(muenzen, maximum):
    """Wie in Notebook 11: tabelle[b] ist die wenigste Münzzahl für den Betrag b."""
    tabelle = [None] * (maximum + 1)
    tabelle[0] = 0
    for betrag in range(1, maximum + 1):
        for muenze in muenzen:
            if muenze <= betrag and tabelle[betrag - muenze] is not None:
                if tabelle[betrag] is None or tabelle[betrag - muenze] + 1 < tabelle[betrag]:
                    tabelle[betrag] = tabelle[betrag - muenze] + 1
    return tabelle

def schlechte_betraege(muenzen, maximum):
    tabelle = wenigste_tabelle(muenzen, maximum)
    schlecht = []
    for betrag in range(1, maximum + 1):
        if len(gierig_muenzen(betrag, muenzen)) != tabelle[betrag]:
            schlecht.append(betrag)
    return schlecht

for muenzen in [(1, 4, 6), (1, 3, 4), (1, 2, 5, 10, 20, 50), (1, 5, 10, 25)]:
    schlecht = schlechte_betraege(muenzen, 200)
    print(muenzen, ':', len(schlecht), 'von 200 Beträgen falsch, die ersten:', schlecht[:6])
```

Mit den Münzen 1, 4, 6 ist das Verfahren bei 65 von 200 Beträgen schlecht (zuerst 8, 9, 14, 15, 20, 21), mit 1, 3, 4 bei 49 (zuerst 6, 10, 14).
Bei den Euro-Münzen 1, 2, 5, 10, 20, 50 und bei 1, 5, 10, 25 (Dollar-Cent) findet der Test **keinen** Fehler. Ein einziger schlechter Betrag genügt,
um das gierige Verfahren zu **widerlegen**: bei 1, 3, 4 etwa der Betrag 6, denn 4 + 1 + 1 hat drei Münzen, 3 + 3 nur zwei.

</details>

#### Frage 1.3

1. Der Test für die Euro-Münzen findet bis 200 keinen schlechten Betrag. Ist damit bewiesen, dass das gierige Verfahren für diese Münzen **immer**
   richtig ist? Und was bedeutet ein schlechter Betrag wie 8 bei den Münzen 1, 4, 6?
2. Warum scheitert das gierige Verfahren bei den Münzen 1, 4, 6 beim Betrag 8? Was ist dabei die „scheinbar beste“ Entscheidung, und was
   übersieht sie?
3. Wie viel Arbeit macht das gierige Verfahren im Vergleich zur Tabelle aus Notebook 11 bei `k` Münzsorten und dem Betrag `B`?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Nein. Ein Test **beweist** nur, dass es bis 200 stimmt. Für die Euro-Münzen lässt sich allgemein beweisen, dass „größte zuerst“ immer richtig ist (mit einem
   Austauschargument, ähnlich wie bei den Terminen gleich); hier verlassen wir uns auf den Test. Umgekehrt
   **widerlegt** schon ein einziger schlechter Betrag das Verfahren: Bei 1, 4, 6 reicht der Betrag 8 (gierig 6 + 1 + 1, besser 4 + 4).
2. Die scheinbar beste Entscheidung ist, die größte Münze 6 zu nehmen: Sie bringt den Betrag am schnellsten nach unten. Der Rest 2 lässt sich
   dann aber nur mit zwei Einern zahlen. Die Entscheidung „6“ lässt sich **nicht mehr rückgängig machen**, und die Alternative 4 + 4 geht
   ohne die 6 auf. Die gierige Wahl **verbaut** hier die beste Lösung.
3. Gierig: **O(k log k)** zum Sortieren der Münzen, dazu eine Runde der `while`-Schleife je gewählter Münze (mit einer Division statt der Schleife wären es
   nur k Schritte). Die Tabelle aus Notebook 11 braucht **O(B · k)**, sie rechnet alle Beträge bis `B` aus. Dafür ist die Tabelle immer richtig. Das ist der
   Preis: Gierig ist schnell, aber man muss begründen, dass es stimmt.

</details>

## 2. Termine planen: welche Regel stimmt?

Ein Raum soll an einem Tag für möglichst **viele** Veranstaltungen genutzt werden. Jede Veranstaltung ist ein Termin `(anfang, ende)` in Stunden,
und zwei Termine dürfen sich nicht **überschneiden**. Hört einer um 5 Uhr auf und der andere fängt um 5 Uhr an, ist das in Ordnung. Zwei Termine
`a` und `b` überlappen also genau dann, wenn `a[0] < b[1]` und `b[0] < a[1]` gilt.

Gesucht ist die **größte** Zahl von Terminen, die sich nicht überlappen. Das gierige Verfahren sortiert die Termine nach einer **Regel** und geht sie
der Reihe nach durch: Passt ein Termin zu den schon gewählten (kein Überlappen), nimmt man ihn. Doch welche Regel? Drei Vorschläge:

- `'ende'`: der Termin, der **am frühesten endet**, zuerst,
- `'laenge'`: der **kürzeste** Termin zuerst,
- `'anfang'`: der Termin, der **am frühesten anfängt**, zuerst.

Alle drei klingen vernünftig. Mit dem Wissen aus Abschnitt 1 prüfst du sie nicht durch Raten, sondern durch **Gegenbeispiele**.

#### Aufgabe 2.1

Schreibe `ueberlappen(a, b)` und `gierig_termine(termine, regel)`. Die Funktion `gierig_termine` sortiert die Termine je nach `regel`
(`'ende'`: nach dem Ende, `'laenge'`: nach der Länge `ende - anfang`, `'anfang'`: nach dem Anfang), geht sie der Reihe nach durch, nimmt jeden, der
mit **keinem** der schon gewählten überlappt, und gibt die Liste der gewählten Termine zurück. Bei gleichem Schlüssel bleibt die Reihenfolge der Eingabe
(`sorted` ist stabil).

**Erwartete Ausgabe:**
```
False True
ende : 4 Termine [(1, 4), (5, 7), (8, 11), (12, 16)]
laenge : 4 Termine [(3, 5), (5, 7), (8, 11), (12, 16)]
anfang : 3 Termine [(0, 6), (6, 10), (12, 16)]
```

In [ ]:
def ueberlappen(a, b):
    return False                             # ersetze diese Zeile

def gierig_termine(termine, regel):
    if regel == 'ende':
        reihenfolge = sorted(termine, key=lambda t: t[1])
    # dein Code: die anderen beiden Regeln
    gewaehlt = []
    # dein Code
    return gewaehlt

TERMINE = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
print(ueberlappen((0, 5), (5, 8)), ueberlappen((0, 5), (4, 8)))
for regel in ['ende', 'laenge', 'anfang']:
    wahl = gierig_termine(TERMINE, regel)
    print(regel, ':', len(wahl), 'Termine', wahl)

<details>
<summary>Tipp anzeigen</summary>

Je nach `regel` ein anderes `key=lambda t: ...` bei `sorted`. Danach: für jeden `termin` aus `reihenfolge` prüfe mit einer Schleife über `gewaehlt`, ob er mit
einem überlappt. Nur wenn nicht, hänge ihn an.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def ueberlappen(a, b):
    return a[0] < b[1] and b[0] < a[1]       # a beginnt vor Ende von b und b vor Ende von a

def gierig_termine(termine, regel):
    if regel == 'ende':
        reihenfolge = sorted(termine, key=lambda t: t[1])
    elif regel == 'laenge':
        reihenfolge = sorted(termine, key=lambda t: t[1] - t[0])
    else:
        reihenfolge = sorted(termine, key=lambda t: t[0])
    gewaehlt = []
    for termin in reihenfolge:
        frei = True
        for andere in gewaehlt:
            if ueberlappen(termin, andere):
                frei = False
        if frei:
            gewaehlt.append(termin)
    return gewaehlt

TERMINE = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
print(ueberlappen((0, 5), (5, 8)), ueberlappen((0, 5), (4, 8)))
for regel in ['ende', 'laenge', 'anfang']:
    wahl = gierig_termine(TERMINE, regel)
    print(regel, ':', len(wahl), 'Termine', wahl)
```

Die Regeln `'ende'` und `'laenge'` wählen beide vier Termine (`'ende'`: `(1, 4)`, `(5, 7)`, `(8, 11)`, `(12, 16)`), die Regel `'anfang'` nur drei:
`(0, 6)`, `(6, 10)`, `(12, 16)`. Auf **diesem** Beispiel ist `'laenge'` also genauso gut wie `'ende'`. Das ist trügerisch: Ein Beispiel, bei dem
das Verfahren klappt, sagt nichts darüber, ob es immer klappt.

</details>

#### Aufgabe 2.2

Die **genaue** Lösung zum Vergleichen: Schreibe `beste_zahl(termine)`. Sie probiert alle Teilmengen aus und gibt die größte Zahl
von Terminen zurück, von denen keine zwei überlappen. `itertools.combinations(termine, k)` liefert alle Auswahlen von `k` Terminen (wie in Notebook 1).
Probiere `k` von der Gesamtzahl herunter bis 1; die erste passende Auswahl ist die größte. Ohne Termine ist die Antwort 0.

**Erwartete Ausgabe:**
```
Größte Zahl überschneidungsfreier Termine: 4
2 2 0
11 Termine, 2048 Teilmengen
```

In [ ]:
import itertools

def ueberlappen(a, b):
    return a[0] < b[1] and b[0] < a[1]

def beste_zahl(termine):
    # dein Code
    return 0

TERMINE = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
print('Größte Zahl überschneidungsfreier Termine:', beste_zahl(TERMINE))
print(beste_zahl([(0, 5), (4, 6), (5, 10)]), beste_zahl([(0, 10), (1, 2), (3, 4)]), beste_zahl([]))
print(len(TERMINE), 'Termine,', 2 ** len(TERMINE), 'Teilmengen')

<details>
<summary>Tipp anzeigen</summary>

Außen `for k in range(len(termine), 0, -1)`, darin `for auswahl in itertools.combinations(termine, k)`. Prüfe mit
`itertools.combinations(auswahl, 2)`, ob ein Paar überlappt. Wenn keines überlappt, `return k`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools

def ueberlappen(a, b):
    return a[0] < b[1] and b[0] < a[1]

def beste_zahl(termine):
    for k in range(len(termine), 0, -1):                       # große Auswahlen zuerst
        for auswahl in itertools.combinations(termine, k):
            geht = True
            for a, b in itertools.combinations(auswahl, 2):    # alle Paare der Auswahl
                if ueberlappen(a, b):
                    geht = False
            if geht:
                return k                                       # die erste gefundene ist die größte
    return 0

TERMINE = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
print('Größte Zahl überschneidungsfreier Termine:', beste_zahl(TERMINE))
print(beste_zahl([(0, 5), (4, 6), (5, 10)]), beste_zahl([(0, 10), (1, 2), (3, 4)]), beste_zahl([]))
print(len(TERMINE), 'Termine,', 2 ** len(TERMINE), 'Teilmengen')
```

Für die 11 Termine gibt es 2 048 Teilmengen, und das Durchprobieren findet 4 als Bestwert. Das gierige Verfahren mit `'ende'` hat also hier 4 geschafft: das Optimum. Die Laufzeit des
Durchprobierens ist aber **O(2ⁿ · n²)**, bei 40 Terminen schon unbrauchbar. Es taugt als **Prüfer** für kleine Beispiele, nicht für große Eingaben.

</details>

#### Aufgabe 2.3

Jetzt suchst du Gegenbeispiele mit Zufall. `zufalls_termine(anzahl)` ist fertig. Schreibe `suche_gegenbeispiel(regel, versuche)`: Sie setzt mit
`random.seed(1)` den Zufall auf einen festen Anfang (damit alle dasselbe sehen), erzeugt `versuche`-mal fünf Zufallstermine, vergleicht
`len(gierig_termine(...))` mit `beste_zahl(...)` und gibt das Paar `(schlecht, erstes)` zurück: Zahl der Fälle, in denen die gierige Regel schlechter war, und das
Paar `(Nummer des Versuchs, Termine)` des **ersten** solchen Falls (oder `None`).

**Erwartete Ausgabe:**
```
ende : 0 von 500 Zufallsfällen schlecht
laenge : 12 von 500 Zufallsfällen schlecht
  erster Fall (Nr. 47 ): [(5, 8), (1, 7), (8, 13), (4, 6), (3, 5)]
  gierig: 2 , beste Lösung: 3
anfang : 85 von 500 Zufallsfällen schlecht
  erster Fall (Nr. 17 ): [(3, 6), (1, 6), (2, 5), (4, 5), (2, 4)]
  gierig: 1 , beste Lösung: 2
```

In [ ]:
import random
import itertools

def ueberlappen(a, b):
    return a[0] < b[1] and b[0] < a[1]

def gierig_termine(termine, regel):
    if regel == 'ende':
        reihenfolge = sorted(termine, key=lambda t: t[1])
    elif regel == 'laenge':
        reihenfolge = sorted(termine, key=lambda t: t[1] - t[0])
    else:
        reihenfolge = sorted(termine, key=lambda t: t[0])
    gewaehlt = []
    for termin in reihenfolge:
        frei = True
        for andere in gewaehlt:
            if ueberlappen(termin, andere):
                frei = False
        if frei:
            gewaehlt.append(termin)
    return gewaehlt

def beste_zahl(termine):
    for k in range(len(termine), 0, -1):
        for auswahl in itertools.combinations(termine, k):
            geht = True
            for a, b in itertools.combinations(auswahl, 2):
                if ueberlappen(a, b):
                    geht = False
            if geht:
                return k
    return 0

def zufalls_termine(anzahl):
    termine = []
    for i in range(anzahl):
        anfang = random.randint(0, 9)
        termine.append((anfang, anfang + random.randint(1, 6)))     # Dauer 1 bis 6 Stunden
    return termine

def suche_gegenbeispiel(regel, versuche):
    random.seed(1)
    schlecht = 0
    erstes = None
    # dein Code
    return schlecht, erstes

for regel in ['ende', 'laenge', 'anfang']:
    schlecht, erstes = suche_gegenbeispiel(regel, 500)
    print(regel, ':', schlecht, 'von 500 Zufallsfällen schlecht')
    if erstes is not None:
        nummer, termine = erstes
        print('  erster Fall (Nr.', nummer, '):', termine)
        print('  gierig:', len(gierig_termine(termine, regel)), ', beste Lösung:', beste_zahl(termine))

<details>
<summary>Tipp anzeigen</summary>

`for nummer in range(1, versuche + 1):` Erzeuge `termine = zufalls_termine(5)` und vergleiche die beiden Zahlen. Bei einem Unterschied zählst du `schlecht` hoch
und merkst dir `(nummer, termine)`, falls `erstes` noch `None` ist.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random
import itertools

def ueberlappen(a, b):
    return a[0] < b[1] and b[0] < a[1]

def gierig_termine(termine, regel):
    if regel == 'ende':
        reihenfolge = sorted(termine, key=lambda t: t[1])
    elif regel == 'laenge':
        reihenfolge = sorted(termine, key=lambda t: t[1] - t[0])
    else:
        reihenfolge = sorted(termine, key=lambda t: t[0])
    gewaehlt = []
    for termin in reihenfolge:
        frei = True
        for andere in gewaehlt:
            if ueberlappen(termin, andere):
                frei = False
        if frei:
            gewaehlt.append(termin)
    return gewaehlt

def beste_zahl(termine):
    for k in range(len(termine), 0, -1):
        for auswahl in itertools.combinations(termine, k):
            geht = True
            for a, b in itertools.combinations(auswahl, 2):
                if ueberlappen(a, b):
                    geht = False
            if geht:
                return k
    return 0

def zufalls_termine(anzahl):
    termine = []
    for i in range(anzahl):
        anfang = random.randint(0, 9)
        termine.append((anfang, anfang + random.randint(1, 6)))     # Dauer 1 bis 6 Stunden
    return termine

def suche_gegenbeispiel(regel, versuche):
    random.seed(1)
    schlecht = 0
    erstes = None
    for nummer in range(1, versuche + 1):
        termine = zufalls_termine(5)
        if len(gierig_termine(termine, regel)) != beste_zahl(termine):
            schlecht = schlecht + 1
            if erstes is None:                         # nur den ersten Fall merken
                erstes = (nummer, termine)
    return schlecht, erstes

for regel in ['ende', 'laenge', 'anfang']:
    schlecht, erstes = suche_gegenbeispiel(regel, 500)
    print(regel, ':', schlecht, 'von 500 Zufallsfällen schlecht')
    if erstes is not None:
        nummer, termine = erstes
        print('  erster Fall (Nr.', nummer, '):', termine)
        print('  gierig:', len(gierig_termine(termine, regel)), ', beste Lösung:', beste_zahl(termine))
```

Die Regel `'ende'` besteht alle 500 Versuche, `'laenge'` fällt bei 12 und `'anfang'` bei 85 durch. Ein Fall genügt als Gegenbeispiel: Bei `'laenge'` im ersten Fall
(Nr. 47, Termine `(5, 8)`, `(1, 7)`, `(8, 13)`, `(4, 6)`, `(3, 5)`) wählt das Verfahren zuerst den kürzesten Termin `(4, 6)`. Er blockiert `(3, 5)` und `(5, 8)`, die zusammen
mit `(8, 13)` drei Termine ergeben hätten, und es bleiben zwei. Für `'ende'` sind 500 bestandene Versuche **kein Beweis**: Wir haben nur **kein** Gegenbeispiel
gefunden. Warum es stimmt, zeigt das Austauschargument gleich.

</details>

#### Frage 2.4

Gegenbeispiele widerlegen eine Regel. Aber warum stimmt `'ende'` (früh endend zuerst)? Führe ein **Austauschargument**:

1. Sei `g` der Termin, der **am frühesten endet** (die erste gierige Wahl). Sei `O` eine **beste** Auswahl, und sei `o` der Termin in `O`,
   der am frühesten endet. Was passiert, wenn man in `O` den Termin `o` durch `g` **ersetzt**? Warum bleibt die Auswahl überschneidungsfrei,
   und wie viele Termine hat sie?
2. Wie folgt daraus, dass die gierige Wahl **richtig** ist, auch für den zweiten, dritten, ... Termin?
3. Zeige an den Terminen `(0, 5)`, `(4, 6)`, `(5, 10)`, dass **das Argument für „kürzester zuerst“ nicht funktioniert**. Wo geht es schief?
4. Welche Laufzeit hat das Verfahren mit der Regel `'ende'`, wenn man nur den Ende-Zeitpunkt des **zuletzt gewählten** Termins merkt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `g` endet höchstens so spät wie `o` (es endet am frühesten von **allen**). Die anderen Termine von `O` beginnen alle erst **nach** dem Ende von `o`
   (sonst würden sie mit `o` überlappen, und `o` endet vor ihnen), also auch nach dem Ende von `g`. `g` überlappt deshalb mit keinem von ihnen. Die Auswahl
   `(O ohne o) + g` ist überschneidungsfrei und hat **genauso viele** Termine wie `O`: Sie ist auch eine beste Auswahl, und sie **enthält `g`**. Man verliert
   durch die gierige Wahl also nichts.
2. Danach schaust du nur noch auf die Termine, die **nach dem Ende von `g`** beginnen: Das ist wieder dieselbe Aufgabe, nur kleiner. Wieder darf man den
   frühesten Termin nehmen, wieder gilt das Argument. Durch Wiederholen (Induktion) folgt, dass die gierige Auswahl **so groß wie eine beste** ist.
3. Der kürzeste Termin ist `(4, 6)`. Er überlappt mit **beiden** anderen, also bleibt nur 1 Termin. Die beste Auswahl `(0, 5)` und `(5, 10)` hat 2.
   Beim Austausch würde man `(4, 6)` an die Stelle von `(0, 5)` setzen, aber `(4, 6)` überlappt auch mit `(5, 10)`: Ein kurzer Termin kann **mitten** in der
   Zeit liegen und dort mehrere blockieren. Beim frühesten **Ende** kann das nicht passieren, denn dann liegt nichts mehr vor ihm.
4. Das Sortieren braucht **O(n log n)**. Danach genügt ein Durchgang: Man merkt sich das Ende des zuletzt gewählten Termins und nimmt jeden Termin, der
   **nicht früher** beginnt (`t[0] >= ende`). Das sind O(n) Schritte, zusammen **O(n log n)**. Unsere Funktion `gierig_termine` ist langsamer
   (sie vergleicht mit allen gewählten Terminen, O(n²)), aber richtig.

</details>

## 3. Finde den Fehler

Zwei typische Fehler bei gierigen Verfahren: die falsche **Reihenfolge** und eine falsche **Prüfung**.

#### Aufgabe 3.1: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def gierig_falsch(betrag, muenzen):
    plan = []
    for muenze in muenzen:                   # die Münzen in der Reihenfolge der Eingabe
        while betrag >= muenze:
            plan.append(muenze)
            betrag = betrag - muenze
    return plan

print(len(gierig_falsch(12, (1, 4, 6))))
print(len(gierig_falsch(12, (6, 4, 1))))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
12
2
```

Ausgegeben werden `12` und `2`. Bei `(1, 4, 6)` kommt die Münze 1 **zuerst** dran und passt in den Betrag 12 zwölfmal: zwölf Einer. Das Verfahren
funktioniert nur, wenn man die Münzen **absteigend** durchgeht. Bei `(6, 4, 1)` stimmt es zufällig, weil die Reihenfolge schon passt. Im Programm macht
`sorted(muenzen, reverse=True)` die Reihenfolge unabhängig von der Eingabe. Die Regel „größte zuerst“ ist ein **Teil** des Verfahrens, vergiss die
Sortierung nicht.

</details>

#### Aufgabe 3.2: Was wird ausgegeben?

Überlege zuerst, was der Code ausgibt, und schreibe es auf. Führe die Zelle erst danach aus.

In [ ]:
def ueberlappen_falsch(a, b):
    return a[0] <= b[1] and b[0] <= a[1]     # <= statt <

def gierig(termine):
    reihenfolge = sorted(termine, key=lambda t: t[1])
    gewaehlt = []
    for termin in reihenfolge:
        frei = True
        for andere in gewaehlt:
            if ueberlappen_falsch(termin, andere):
                frei = False
        if frei:
            gewaehlt.append(termin)
    return gewaehlt

print(gierig([(0, 5), (5, 10), (10, 12)]))
print(gierig([(0, 5), (6, 10), (11, 12)]))

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

**Ausgabe:**
```
[(0, 5), (10, 12)]
[(0, 5), (6, 10), (11, 12)]
```

Ausgegeben werden `[(0, 5), (10, 12)]` und `[(0, 5), (6, 10), (11, 12)]`. Nach dem Ende sortiert kommen `(0, 5)`, `(5, 10)`, `(10, 12)`. Der erste wird gewählt. Beim zweiten
ist `0 <= 10 and 5 <= 5` wahr, er gilt als überlappend und wird abgelehnt, obwohl er genau dort beginnt, wo `(0, 5)` endet. Der dritte wird nur mit `(0, 5)` verglichen
(der abgelehnte steht nicht in `gewaehlt`) und gewählt. Es sind 2 statt 3 Termine. Beim zweiten Aufruf berühren sich keine Termine, dort stimmt das Ergebnis. **Ob** Termine,
die sich nur in einem Zeitpunkt berühren, als überlappend gelten, legt die Aufgabe fest. Lies es genau und teste die **Grenzfälle** (Ende gleich Anfang).

</details>

## Wann darf man gierig sein?

Eine Checkliste für gierige Verfahren in der 1. Runde:

1. **Welche Reihenfolge?** Typisch: größte zuerst, früh endend zuerst, billigste zuerst. Mit `sorted(..., key=lambda ...)` herstellen.
2. **Gegenbeispiele suchen**: für kleine Eingaben das gierige Verfahren mit **Durchprobieren** (oder einer Tabelle wie in Notebook 11) vergleichen, am besten mit vielen
   Zufallseingaben. Ein Unterschied widerlegt das Verfahren.
3. **Begründen**: ein **Austauschargument** („Nimm eine beste Lösung. Ersetze ihren ersten Teil durch die gierige Wahl. Sie wird nicht schlechter.“)
   oder ein anderer Beweis. Ein bestandener Test beweist nichts, und in der Dokumentation muss die Begründung stehen.
4. **Wenn es nicht stimmt**: Tabelle (Notebook 10 und 11) oder Durchprobieren mit Schranken (Notebook 3).

## Wettbewerbs-Ecke 1: Urlaubsfahrt

*Angelehnt an Aufgabe 4 „Urlaubsfahrt“ der 1. Runde des 38. Bundeswettbewerbs Informatik (2019/20).*

Ein Wohnmobil fährt eine Strecke von `laenge` Kilometern. Es verbraucht `verbrauch` Liter Diesel auf 100 km, der Tank fasst `tank` Liter, am Start sind `anfang`
Liter darin. Auf der Strecke liegen Tankstellen: Zu jeder gehören die **Entfernung vom Start** und der **Dieselpreis** in Cent pro Liter. Die Familie will
**so selten wie möglich** tanken und unter allen Plänen mit so wenigen Stopps den **billigsten** wählen. Am Ziel darf der Tank leer sein.

Damit alles ganzzahlig bleibt, rechnen wir den Diesel in **Hundertstel Litern**: Dann verbraucht jeder Kilometer genau `verbrauch` Einheiten, `lies_fahrt`
liest die Datei und rechnet `tank` und `anfang` schon um. Die Dateien `daten/fahrt1.txt` bis `fahrt5.txt` sind die Beispiele der Aufgabe. Wir lösen die Aufgabe in
drei gierigen Schritten und prüfen sie mit Durchprobieren:

- **Schritt 1, wenige Stopps:** Fahre immer bis zur **am weitesten entfernten** Tankstelle, die du noch erreichst, und tanke dort **voll**.
- **Schritt 2, wie viel tanken:** Sind die **Stopps** festgelegt, gilt an jedem Stopp: Suche den nächsten Ort (Stopp oder das Ziel mit Preis 0), dessen Preis
  **nicht höher** ist als hier. Liegt er in Reichweite einer vollen Tankfüllung, tanke nur so viel, dass du gerade dorthin kommst (oder gar nichts, wenn der Tank dafür schon reicht).
  Sonst fülle den Tank ganz auf.
- **Schritt 3, welche Stopps:** Hier hilft Gier nicht allein. Wir probieren **alle** Auswahlen von Tankstellen mit der kleinsten Stoppzahl durch.

#### Aufgabe E.1

Schreibe `wenigste_stopps(fahrt)`: Sie gibt die Liste der Kilometerstände der Stopps zurück, die dieses gierige Verfahren wählt, oder `None`, wenn die
Fahrt nicht möglich ist. Mit `d` Kilometern kommt man mit `menge` Einheiten im Tank, wenn `d * verbrauch <= menge` gilt, also `d <= menge // verbrauch`. `weite` ist
die Stelle, bis zu der man im Moment kommt: am Start `anfang // verbrauch`, nach einem vollen Tanken an der Stelle `position` ist es
`position + tank // verbrauch`. Solange `weite < laenge` gilt, wähle die weiteste Tankstelle `d` mit `position < d <= weite`.

**Erwartete Ausgabe:**
```
fahrt1 : 2 Stopps, die ersten: [100, 400]
fahrt2 : 9 Stopps, die ersten: [1118, 2240, 3346]
fahrt3 : 1 Stopps, die ersten: [465]
fahrt4 : 2 Stopps, die ersten: [344, 706]
fahrt5 : 9 Stopps, die ersten: [194, 1305, 2433]
```

In [ ]:
import itertools
import math

def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    verbrauch, tank, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        stationen.append((zahlen[5 + 2 * i], zahlen[6 + 2 * i]))      # (Entfernung, Preis)
    stationen.sort()
    return verbrauch, tank * 100, anfang * 100, laenge, stationen       # Diesel in Hundertstel Litern

FAHRTEN = [lies_fahrt(f'daten/fahrt{i}.txt') for i in range(1, 6)]

def wenigste_stopps(fahrt):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    position = 0
    weite = anfang // verbrauch
    stopps = []
    # dein Code
    return stopps

for nummer in range(5):
    stopps = wenigste_stopps(FAHRTEN[nummer])
    print('fahrt' + str(nummer + 1), ':', len(stopps), 'Stopps, die ersten:', stopps[:3])

<details>
<summary>Tipp anzeigen</summary>

`while weite < laenge:` Sammle in `kandidaten` alle Entfernungen `d` mit `position < d <= weite`. Ist die Liste leer, `return None`. Sonst `position = max(kandidaten)`,
hänge `position` an `stopps` an und setze `weite = position + tank // verbrauch`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools
import math

def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    verbrauch, tank, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        stationen.append((zahlen[5 + 2 * i], zahlen[6 + 2 * i]))      # (Entfernung, Preis)
    stationen.sort()
    return verbrauch, tank * 100, anfang * 100, laenge, stationen       # Diesel in Hundertstel Litern

FAHRTEN = [lies_fahrt(f'daten/fahrt{i}.txt') for i in range(1, 6)]

def wenigste_stopps(fahrt):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    position = 0
    weite = anfang // verbrauch                # so weit kommt man mit der Anfangsfüllung
    stopps = []
    while weite < laenge:
        kandidaten = [d for d, preis in stationen if position < d <= weite]
        if len(kandidaten) == 0:
            return None                        # keine Tankstelle in Reichweite
        position = max(kandidaten)             # gierig: die weiteste
        stopps.append(position)
        weite = position + tank // verbrauch   # dort voll getankt
    return stopps

for nummer in range(5):
    stopps = wenigste_stopps(FAHRTEN[nummer])
    print('fahrt' + str(nummer + 1), ':', len(stopps), 'Stopps, die ersten:', stopps[:3])
```

Die Zahl der Stopps ist 2, 9, 1, 2 und 9 (dieselben Zahlen wie im Lösungsnotebook zur Aufgabe). Hier lässt sich das **Austauschargument** gut
führen: Hat eine beste Fahrt ihren ersten Stopp an einer Tankstelle `a`, die **vor** der weitesten erreichbaren Tankstelle `g` liegt, dann ersetze `a` durch `g`. Mit einem vollen Tank kommt man
von `g` mindestens so weit wie von `a`, denn `g` liegt weiter vorn. Der Rest der Fahrt bleibt möglich, die Stoppzahl wird nicht größer. Genauso beim zweiten Stopp und so weiter.

</details>

#### Aufgabe E.2

Schreibe `tank_kosten(fahrt, stopps)`: Die Stopps (Liste der Entfernungen, aufsteigend) sind festgelegt. Die Funktion tankt nach der Regel aus
Schritt 2 und gibt das Paar `(kosten, kaeufe)` zurück: `kosten` in Einheiten (Hundertstel Liter mal Cent, durch 10 000 geteilt sind es Euro) und die Liste
der Käufe `(km, menge)`. Ist die Fahrt mit diesen Stopps nicht möglich (eine Strecke ist weiter, als der Tank reicht), gib `None` zurück. Die Anfangsfüllung kostet nichts.
Das Gerüst legt schon die Orte (Start, Stopps, Ziel) und ihre Preise an (der Start hat keinen Preis, das Ziel hat 0).

**Erwartete Ausgabe:**
```
(817000, [(100, 1000), (400, 4800)])
None
81.70 Euro
  bei km 100 : 10.0 Liter
  bei km 400 : 48.0 Liter
fahrt3 [465] : 99.20 Euro
fahrt4 [344, 706] : 243.22 Euro
fahrt4 [264, 607] : 236.88 Euro
```

In [ ]:
import itertools
import math

def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    verbrauch, tank, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        stationen.append((zahlen[5 + 2 * i], zahlen[6 + 2 * i]))      # (Entfernung, Preis)
    stationen.sort()
    return verbrauch, tank * 100, anfang * 100, laenge, stationen       # Diesel in Hundertstel Litern

FAHRTEN = [lies_fahrt(f'daten/fahrt{i}.txt') for i in range(1, 6)]

def tank_kosten(fahrt, stopps):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    preis_bei = {}
    for d, p in stationen:
        preis_bei[d] = p
    orte = [0] + stopps + [laenge]
    preise = [None] + [preis_bei[d] for d in stopps] + [0]
    inhalt = anfang
    kosten = 0
    kaeufe = []
    # dein Code: für i in range(len(orte) - 1): an einem Stopp (i > 0) tanken, dann zum nächsten Ort fahren
    return kosten, kaeufe

print(tank_kosten(FAHRTEN[0], [100, 400]))
print(tank_kosten(FAHRTEN[0], [100, 900]))
kosten, kaeufe = tank_kosten(FAHRTEN[0], [100, 400])
print(f'{kosten / 10000:.2f} Euro')
for km, menge in kaeufe:
    print('  bei km', km, ':', menge / 100, 'Liter')
for nummer, stopps in [(3, [465]), (4, [344, 706]), (4, [264, 607])]:
    kosten, kaeufe = tank_kosten(FAHRTEN[nummer - 1], stopps)
    print('fahrt' + str(nummer), stopps, ':', f'{kosten / 10000:.2f} Euro')

<details>
<summary>Tipp anzeigen</summary>

Für jedes `i`: Ist `i > 0`, suche mit `j = i + 1` und `while preise[j] > preise[i]: j = j + 1` den nächsten Ort mit höchstens demselben Preis.
`bis_dahin = (orte[j] - orte[i]) * verbrauch`. Ist `bis_dahin <= tank`, kaufe `max(0, bis_dahin - inhalt)`, sonst `tank - inhalt`. Danach fährst du zu `orte[i + 1]`: Brauchst du
mehr, als im Tank ist, `return None`, sonst ziehe es vom Inhalt ab.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools
import math

def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    verbrauch, tank, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        stationen.append((zahlen[5 + 2 * i], zahlen[6 + 2 * i]))      # (Entfernung, Preis)
    stationen.sort()
    return verbrauch, tank * 100, anfang * 100, laenge, stationen       # Diesel in Hundertstel Litern

FAHRTEN = [lies_fahrt(f'daten/fahrt{i}.txt') for i in range(1, 6)]

def tank_kosten(fahrt, stopps):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    preis_bei = {}
    for d, p in stationen:
        preis_bei[d] = p
    orte = [0] + stopps + [laenge]
    preise = [None] + [preis_bei[d] for d in stopps] + [0]
    inhalt = anfang
    kosten = 0
    kaeufe = []
    for i in range(len(orte) - 1):
        if i > 0:                                          # an einem Stopp: tanken
            j = i + 1
            while preise[j] > preise[i]:                   # nächster Ort, der nicht teurer ist (spätestens das Ziel)
                j = j + 1
            bis_dahin = (orte[j] - orte[i]) * verbrauch
            if bis_dahin <= tank:
                kauf = max(0, bis_dahin - inhalt)          # nur so viel, dass man gerade dorthin kommt
            else:
                kauf = tank - inhalt                       # sonst voll
            inhalt = inhalt + kauf
            kosten = kosten + kauf * preise[i]
            if kauf > 0:
                kaeufe.append((orte[i], kauf))
        noetig = (orte[i + 1] - orte[i]) * verbrauch
        if noetig > inhalt:
            return None                                    # reicht nicht bis zum nächsten Ort
        inhalt = inhalt - noetig
    return kosten, kaeufe

print(tank_kosten(FAHRTEN[0], [100, 400]))
print(tank_kosten(FAHRTEN[0], [100, 900]))
kosten, kaeufe = tank_kosten(FAHRTEN[0], [100, 400])
print(f'{kosten / 10000:.2f} Euro')
for km, menge in kaeufe:
    print('  bei km', km, ':', menge / 100, 'Liter')
for nummer, stopps in [(3, [465]), (4, [344, 706]), (4, [264, 607])]:
    kosten, kaeufe = tank_kosten(FAHRTEN[nummer - 1], stopps)
    print('fahrt' + str(nummer), stopps, ':', f'{kosten / 10000:.2f} Euro')
```

Beispiel 1 mit den Stopps 100 und 400 kostet 817 000 Einheiten, das sind 81,70 Euro: 10 Liter bei km 100 zu 145 Cent (am Start sind schon 22 Liter im Tank) und 48 Liter
bei km 400 zu 140 Cent, die genau bis zum Ziel reichen. Mit den Stopps 100 und 900 geht es nicht: 800 km brauchen 64 Liter, der Tank fasst 55. Beispiel 3 mit dem Stopp
bei 465 kostet 99,20 Euro. Beispiel 4 mit den **gierigen Stopps** 344 und 706 kostet 243,22 Euro, mit den Stopps 264 und 607 nur 236,88 Euro.
Die gierige Regel für die **Menge** ist also in Ordnung, aber die Stopps aus Schritt 1 sind nicht automatisch die billigsten.

</details>

#### Aufgabe E.3

Schreibe `beste_plaene(fahrt)`: Mit `m = len(wenigste_stopps(fahrt))` probiert sie **alle** Auswahlen von `m` Tankstellen durch
(`itertools.combinations(entfernungen, m)`), wertet jede mit `tank_kosten` aus und gibt `(beste, bester, moeglich, machbar)` zurück: die kleinsten Kosten, die zugehörige
Auswahl (als Liste), die Zahl der Auswahlen und die Zahl der Auswahlen, bei denen die Fahrt möglich ist. Das Gerüst enthält
`wenigste_stopps` und `tank_kosten` von oben. Beispiele 2 und 5 (154 und 171 Tankstellen, 9 Stopps) sind zu groß: Dort zeigt die letzte Schleife nur die Zahl der Auswahlen.

**Erwartete Ausgabe:**
```
fahrt1 : 2 Stopps, 3 Auswahlen, 1 machbar
  billigster Plan [100, 400] : 81.70 Euro , gierige Stopps: 81.70 Euro
fahrt3 : 1 Stopps, 14 Auswahlen, 4 machbar
  billigster Plan [465] : 99.20 Euro , gierige Stopps: 99.20 Euro
fahrt4 : 2 Stopps, 136 Auswahlen, 10 machbar
  billigster Plan [264, 607] : 236.88 Euro , gierige Stopps: 243.22 Euro
fahrt2 : 154 Tankstellen, 9 Stopps, 105798396965030 Auswahlen
fahrt5 : 171 Tankstellen, 9 Stopps, 278110855548955 Auswahlen
```

In [ ]:
import itertools
import math

def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    verbrauch, tank, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        stationen.append((zahlen[5 + 2 * i], zahlen[6 + 2 * i]))      # (Entfernung, Preis)
    stationen.sort()
    return verbrauch, tank * 100, anfang * 100, laenge, stationen       # Diesel in Hundertstel Litern

FAHRTEN = [lies_fahrt(f'daten/fahrt{i}.txt') for i in range(1, 6)]

def wenigste_stopps(fahrt):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    position = 0
    weite = anfang // verbrauch
    stopps = []
    while weite < laenge:
        kandidaten = [d for d, preis in stationen if position < d <= weite]
        if len(kandidaten) == 0:
            return None
        position = max(kandidaten)
        stopps.append(position)
        weite = position + tank // verbrauch
    return stopps

def tank_kosten(fahrt, stopps):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    preis_bei = {}
    for d, p in stationen:
        preis_bei[d] = p
    orte = [0] + stopps + [laenge]
    preise = [None] + [preis_bei[d] for d in stopps] + [0]
    inhalt = anfang
    kosten = 0
    kaeufe = []
    for i in range(len(orte) - 1):
        if i > 0:
            j = i + 1
            while preise[j] > preise[i]:
                j = j + 1
            bis_dahin = (orte[j] - orte[i]) * verbrauch
            if bis_dahin <= tank:
                kauf = max(0, bis_dahin - inhalt)
            else:
                kauf = tank - inhalt
            inhalt = inhalt + kauf
            kosten = kosten + kauf * preise[i]
            if kauf > 0:
                kaeufe.append((orte[i], kauf))
        noetig = (orte[i + 1] - orte[i]) * verbrauch
        if noetig > inhalt:
            return None
        inhalt = inhalt - noetig
    return kosten, kaeufe

def beste_plaene(fahrt):
    m = len(wenigste_stopps(fahrt))
    entfernungen = [d for d, preis in fahrt[4]]
    beste = 10 ** 15                                       # größer als alle Kosten
    bester = None
    moeglich = 0
    machbar = 0
    # dein Code
    return beste, bester, moeglich, machbar

for nummer in [1, 3, 4]:
    fahrt = FAHRTEN[nummer - 1]
    m = len(wenigste_stopps(fahrt))
    beste, bester, moeglich, machbar = beste_plaene(fahrt)
    gierig = tank_kosten(fahrt, wenigste_stopps(fahrt))[0]
    print('fahrt' + str(nummer), ':', m, 'Stopps,', moeglich, 'Auswahlen,', machbar, 'machbar')
    print('  billigster Plan', bester, ':', f'{beste / 10000:.2f} Euro', ', gierige Stopps:', f'{gierig / 10000:.2f} Euro')
for nummer in [2, 5]:
    fahrt = FAHRTEN[nummer - 1]
    m = len(wenigste_stopps(fahrt))
    print('fahrt' + str(nummer), ':', len(fahrt[4]), 'Tankstellen,', m, 'Stopps,', math.comb(len(fahrt[4]), m), 'Auswahlen')

<details>
<summary>Tipp anzeigen</summary>

`for auswahl in itertools.combinations(entfernungen, m):` Zähle `moeglich` hoch, rufe `tank_kosten(fahrt, list(auswahl))` auf. Ist das Ergebnis nicht `None`,
zähle `machbar` hoch und merke dir `ergebnis[0]` und die Auswahl, falls `ergebnis[0] < beste` gilt (`beste` beginnt bei einer sehr großen Zahl).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools
import math

def lies_fahrt(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        zahlen = [int(wort) for wort in datei.read().split()]
    verbrauch, tank, anfang, laenge, anzahl = zahlen[:5]
    stationen = []
    for i in range(anzahl):
        stationen.append((zahlen[5 + 2 * i], zahlen[6 + 2 * i]))      # (Entfernung, Preis)
    stationen.sort()
    return verbrauch, tank * 100, anfang * 100, laenge, stationen       # Diesel in Hundertstel Litern

FAHRTEN = [lies_fahrt(f'daten/fahrt{i}.txt') for i in range(1, 6)]

def wenigste_stopps(fahrt):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    position = 0
    weite = anfang // verbrauch
    stopps = []
    while weite < laenge:
        kandidaten = [d for d, preis in stationen if position < d <= weite]
        if len(kandidaten) == 0:
            return None
        position = max(kandidaten)
        stopps.append(position)
        weite = position + tank // verbrauch
    return stopps

def tank_kosten(fahrt, stopps):
    verbrauch, tank, anfang, laenge, stationen = fahrt
    preis_bei = {}
    for d, p in stationen:
        preis_bei[d] = p
    orte = [0] + stopps + [laenge]
    preise = [None] + [preis_bei[d] for d in stopps] + [0]
    inhalt = anfang
    kosten = 0
    kaeufe = []
    for i in range(len(orte) - 1):
        if i > 0:
            j = i + 1
            while preise[j] > preise[i]:
                j = j + 1
            bis_dahin = (orte[j] - orte[i]) * verbrauch
            if bis_dahin <= tank:
                kauf = max(0, bis_dahin - inhalt)
            else:
                kauf = tank - inhalt
            inhalt = inhalt + kauf
            kosten = kosten + kauf * preise[i]
            if kauf > 0:
                kaeufe.append((orte[i], kauf))
        noetig = (orte[i + 1] - orte[i]) * verbrauch
        if noetig > inhalt:
            return None
        inhalt = inhalt - noetig
    return kosten, kaeufe

def beste_plaene(fahrt):
    m = len(wenigste_stopps(fahrt))
    entfernungen = [d for d, preis in fahrt[4]]
    beste = 10 ** 15                                       # größer als alle Kosten
    bester = None
    moeglich = 0
    machbar = 0
    for auswahl in itertools.combinations(entfernungen, m):
        moeglich = moeglich + 1
        ergebnis = tank_kosten(fahrt, list(auswahl))
        if ergebnis is not None:                           # die Fahrt geht mit diesen Stopps
            machbar = machbar + 1
            if ergebnis[0] < beste:
                beste = ergebnis[0]
                bester = list(auswahl)
    return beste, bester, moeglich, machbar

for nummer in [1, 3, 4]:
    fahrt = FAHRTEN[nummer - 1]
    m = len(wenigste_stopps(fahrt))
    beste, bester, moeglich, machbar = beste_plaene(fahrt)
    gierig = tank_kosten(fahrt, wenigste_stopps(fahrt))[0]
    print('fahrt' + str(nummer), ':', m, 'Stopps,', moeglich, 'Auswahlen,', machbar, 'machbar')
    print('  billigster Plan', bester, ':', f'{beste / 10000:.2f} Euro', ', gierige Stopps:', f'{gierig / 10000:.2f} Euro')
for nummer in [2, 5]:
    fahrt = FAHRTEN[nummer - 1]
    m = len(wenigste_stopps(fahrt))
    print('fahrt' + str(nummer), ':', len(fahrt[4]), 'Tankstellen,', m, 'Stopps,', math.comb(len(fahrt[4]), m), 'Auswahlen')
```

Die billigsten Pläne kosten 81,70 Euro (Beispiel 1), 99,20 Euro (Beispiel 3) und 236,88 Euro (Beispiel 4, Stopps 264 und 607): Das sind genau die Zahlen aus
dem Lösungsnotebook zur Aufgabe, das ein ganz anderes Verfahren benutzt. Die gierigen Stopps aus Schritt 1 sind bei den Beispielen 1 und 3 schon die billigsten, bei Beispiel 4 sind sie
um 6,34 Euro teurer. Von 136 Auswahlen mit zwei Stopps sind nur 10 überhaupt machbar. Bei den Beispielen 2 und 5 gibt es rund **106 Billionen** beziehungsweise **278 Billionen** Auswahlen: Durchprobieren geht nicht mehr. Dort braucht man eine Tabelle über die Tankstellen, wie im Lösungsnotebook, oder
bessere Schranken (Notebook 3 und 10). Ein Plan mit einem **gierigen Schritt 1** liefert aber immer die richtige **Zahl** der Stopps (siehe E.1).

</details>

#### Frage E.4

1. In E.3 hat sich gezeigt, dass die weitesten Stopps (Schritt 1) nicht immer die billigsten sind. Warum widerspricht das nicht dem
   Austauschargument aus E.1?
2. Begründe die **Tankregel** aus Schritt 2 mit einem Austausch: Warum ist es nie schlecht, an einem Stopp nur so viel zu tanken, dass man den nächsten
   Ort mit höchstens gleichem Preis gerade erreicht? Und warum voll, wenn kein solcher Ort in Reichweite ist?
3. Das Durchprobieren in E.3 prüft bei Beispiel 4 136 Auswahlen. Wie viele wären es bei `n` Tankstellen und `m` Stopps (Formel), und wie wächst das?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Das Austauschargument aus E.1 zeigt nur, dass die **Zahl** der Stopps minimal ist: Ein früher Stopp darf durch einen späteren ersetzt werden, ohne dass die
   Stoppzahl wächst. Es sagt nichts über die **Kosten**: Die weiteste Tankstelle kann teuer sein, und die billigste liegt vielleicht davor. Die Aufgabe hat
   **zwei** Ziele nacheinander (erst wenige Stopps, dann billig), und nur das erste löst das gierige Verfahren.
2. Gibt es einen Ort `j` in Reichweite, der nicht teurer ist, dann kann man an `j` genauso gut (oder besser) tanken wie hier. Jeder Liter, den man hier **mehr** kauft,
   als man bis `j` braucht, hätte man an `j` höchstens zum selben Preis bekommen: Man tauscht den teureren Liter von hier gegen einen Liter von `j`, und die Fahrt
   bleibt möglich, die Kosten steigen nicht. Gibt es keinen, ist jeder Ort in Reichweite **teurer** als hier. Dann ist hier der billigste Diesel: Jeder Liter, den man später kauft,
   kostet mehr, also füllt man hier so viel ein, wie in den Tank passt.
3. Es sind **n über m** Auswahlen (`math.comb(n, m)`): für 17 Tankstellen und 2 Stopps 136, für 154 Tankstellen und 9 Stopps rund 106 Billionen. Das wächst etwa wie
   **nᵐ**, also sehr schnell mit `m`. Dafür ist die Rechnung je Auswahl klein, sie hilft aber nicht bei vielen Tankstellen.

</details>

## Wettbewerbs-Ecke 2: Frühsport (für Schnelle)

*Angelehnt an Aufgabe 1 „Frühsport“ der 1. Runde des 31. Bundeswettbewerbs Informatik (2012/13).*

Die Schlange Digitail besteht aus Ringeln, die golden (`'g'`) oder rot (`'r'`) sind. Zwei **benachbarte** Ringel dürfen ihre Farben tauschen, das ist **ein**
Schritt. Für einen größeren Schritt der Schlange braucht man an bestimmten Stellen bestimmte Farben: Zum Beispiel müssen die Ringel an den Plätzen `start` bis
`start + anzahl - 1` **golden** sein. Man darf dafür beliebige goldene Ringel nehmen, aber jeder Tausch kostet einen Schritt, und die Schritte sollen
**möglichst wenige** sein. Das Lösungsnotebook zur Aufgabe benutzt dafür ein gieriges Verfahren; hier prüfst du es.

Beispiel `farben = 'rggrgrrgg'`, gesucht werden zwei goldene Ringel an den Plätzen 0 und 1. Die goldenen Ringel liegen an den Plätzen 1, 2, 4, 7, 8. Nimmt man die Ringel
an den Plätzen 1 und 2 und schiebt sie auf 0 und 1, kostet das 2 Tausche (jedes wandert um einen Platz). Nimmt man 1 und 4, wandert das erste um 1 und das zweite von Platz 4 auf
Platz 1 um 3: zusammen 4. Die gierige Idee: Nimm die Ringel der gesuchten Farbe, die **insgesamt am wenigsten weit** wandern müssen.

#### Aufgabe F.1

Eine Auswahl ist eine aufsteigende Liste `auswahl` von Plätzen der Ringel, die man auf die Plätze `start`, `start + 1`, ... bringen will. Der erste gewählte Ringel
geht nach `start`, der zweite nach `start + 1` und so weiter (gleichfarbige Ringel überholen sich nie, das wäre verschwendet). Schreibe `tauschkosten(auswahl, start)`:
die Zahl der Tauschschritte, also die Summe der Wege `abs(platz - ziel)`.

**Erwartete Ausgabe:**
```
2 4 10
1
```

In [ ]:
import itertools

def finde_kandidaten(farben, farbe):
    return [j for j in range(len(farben)) if farben[j] == farbe]

def tauschkosten(auswahl, start):
    summe = 0
    # dein Code
    return summe

print(tauschkosten([1, 2], 0), tauschkosten([1, 4], 0), tauschkosten([4, 7, 8], 2))
print(tauschkosten([3, 5], 4))

<details>
<summary>Tipp anzeigen</summary>

`for i in range(len(auswahl)):` Das Ziel des `i`-ten Ringels ist `start + i`. Addiere `abs(auswahl[i] - ziel)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools

def finde_kandidaten(farben, farbe):
    return [j for j in range(len(farben)) if farben[j] == farbe]

def tauschkosten(auswahl, start):
    summe = 0
    for i in range(len(auswahl)):
        ziel = start + i                       # der i-te gewählte Ringel soll nach start + i
        summe = summe + abs(auswahl[i] - ziel)
    return summe

print(tauschkosten([1, 2], 0), tauschkosten([1, 4], 0), tauschkosten([4, 7, 8], 2))
print(tauschkosten([3, 5], 4))
```

`[1, 2]` nach 0: zweimal ein Platz, also 2; `[1, 4]`: 1 + 3 = 4; `[4, 7, 8]` nach 2, 3, 4: 2 + 4 + 4 = 10; `[3, 5]` nach 4, 5: 1 + 0 = 1 (einer wandert **nach rechts**).

</details>

#### Aufgabe F.2

Schreibe `beste_auswahl(farben, start, anzahl, farbe)`. Sie bestimmt die Plätze `kandidaten` der Ringel mit dieser Farbe (`finde_kandidaten`) und probiert alle **Fenster**
aus `anzahl` **aufeinanderfolgenden** Kandidaten (`kandidaten[a:a + anzahl]`). Sie gibt `(auswahl, kosten)` mit den kleinsten Kosten zurück (bei Gleichstand das erste
Fenster). Gibt es zu wenige Ringel dieser Farbe, gib `(None, None)` zurück.

**Erwartete Ausgabe:**
```
([1, 2], 2)
([4, 7], 2)
([0, 3, 5], 5)
(None, None)
```

In [ ]:
import itertools

def finde_kandidaten(farben, farbe):
    return [j for j in range(len(farben)) if farben[j] == farbe]

def tauschkosten(auswahl, start):
    summe = 0
    for i in range(len(auswahl)):
        summe = summe + abs(auswahl[i] - (start + i))
    return summe

def beste_auswahl(farben, start, anzahl, farbe):
    kandidaten = finde_kandidaten(farben, farbe)
    beste = None
    beste_kosten = None
    # dein Code
    return beste, beste_kosten

print(beste_auswahl('rggrgrrgg', 0, 2, 'g'))
print(beste_auswahl('rggrgrrgg', 5, 2, 'g'))
print(beste_auswahl('rggrgrrgg', 0, 3, 'r'))
print(beste_auswahl('rggrgrrgg', 0, 6, 'g'))

<details>
<summary>Tipp anzeigen</summary>

`for a in range(len(kandidaten) - anzahl + 1):` Ist `anzahl` größer als `len(kandidaten)`, läuft die Schleife nie, und `(None, None)` bleibt stehen.
Das Fenster ist `kandidaten[a:a + anzahl]`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools

def finde_kandidaten(farben, farbe):
    return [j for j in range(len(farben)) if farben[j] == farbe]

def tauschkosten(auswahl, start):
    summe = 0
    for i in range(len(auswahl)):
        summe = summe + abs(auswahl[i] - (start + i))
    return summe

def beste_auswahl(farben, start, anzahl, farbe):
    kandidaten = finde_kandidaten(farben, farbe)
    beste = None
    beste_kosten = None
    for a in range(len(kandidaten) - anzahl + 1):
        auswahl = kandidaten[a:a + anzahl]                 # anzahl benachbarte Kandidaten
        kosten = tauschkosten(auswahl, start)
        if beste_kosten is None or kosten < beste_kosten:
            beste = auswahl
            beste_kosten = kosten
    return beste, beste_kosten

print(beste_auswahl('rggrgrrgg', 0, 2, 'g'))
print(beste_auswahl('rggrgrrgg', 5, 2, 'g'))
print(beste_auswahl('rggrgrrgg', 0, 3, 'r'))
print(beste_auswahl('rggrgrrgg', 0, 6, 'g'))
```

Für zwei goldene Ringel auf den Plätzen 0 und 1 sind es die Kandidaten 1 und 2 mit Kosten 2. Für die Plätze 5 und 6 sind es die Kandidaten 4 und 7 mit Kosten 2
(4 wandert einen Platz nach rechts, 7 einen nach links), das Fenster `[7, 8]` würde 4 kosten. Für drei rote Ringel ab Platz 0 sind es 0, 3 und 5 mit Kosten 5. Gibt es zu
wenige Ringel der Farbe (sechs goldene gibt es nicht), ist das Ergebnis `(None, None)`.

</details>

#### Aufgabe F.3

Prüfe das Fenster-Verfahren **ohne Beweis** mit Durchprobieren. `durchprobiert` (alle Auswahlen) steht im Gerüst. Schreibe `vergleiche_fs(laenge)`: Sie geht durch alle
Farbfolgen aus `'g'` und `'r'` der Länge `laenge` (`itertools.product('gr', repeat=laenge)`), durch alle `anzahl` von 1 bis 3, alle `start` von 0 bis `laenge - anzahl` und beide Farben.
Fälle mit weniger Ringeln der Farbe als `anzahl` überspringst du. Sie gibt `(alle, gleich, naiv_falsch)` zurück: Zahl der Fälle, wie oft `beste_auswahl` dieselben
Kosten wie `durchprobiert` hat, und wie oft die **naive** Auswahl „die ersten `anzahl` Ringel der Farbe von links“ (`kandidaten[:anzahl]`) **teurer** ist als die beste.

**Erwartete Ausgabe:**
```
10166 Fälle, 10166 mal Fenster = beste Auswahl, 5786 mal ist "die ersten von links" schlechter
```

In [ ]:
import itertools

def finde_kandidaten(farben, farbe):
    return [j for j in range(len(farben)) if farben[j] == farbe]

def tauschkosten(auswahl, start):
    summe = 0
    for i in range(len(auswahl)):
        summe = summe + abs(auswahl[i] - (start + i))
    return summe

def beste_auswahl(farben, start, anzahl, farbe):
    kandidaten = finde_kandidaten(farben, farbe)
    beste = None
    beste_kosten = None
    for a in range(len(kandidaten) - anzahl + 1):
        auswahl = kandidaten[a:a + anzahl]
        kosten = tauschkosten(auswahl, start)
        if beste_kosten is None or kosten < beste_kosten:
            beste = auswahl
            beste_kosten = kosten
    return beste, beste_kosten

def durchprobiert(farben, start, anzahl, farbe):
    """Kleinste Kosten über ALLE Auswahlen von anzahl Ringeln dieser Farbe."""
    kandidaten = finde_kandidaten(farben, farbe)
    beste_kosten = None
    for auswahl in itertools.combinations(kandidaten, anzahl):
        kosten = tauschkosten(list(auswahl), start)
        if beste_kosten is None or kosten < beste_kosten:
            beste_kosten = kosten
    return beste_kosten

def vergleiche_fs(laenge):
    alle = 0
    gleich = 0
    naiv_falsch = 0
    # dein Code
    return alle, gleich, naiv_falsch

alle, gleich, naiv_falsch = vergleiche_fs(8)
print(alle, 'Fälle,', gleich, 'mal Fenster = beste Auswahl,', naiv_falsch, 'mal ist "die ersten von links" schlechter')

<details>
<summary>Tipp anzeigen</summary>

Vier verschachtelte Schleifen: über `folge in itertools.product('gr', repeat=laenge)` (mit `farben = ''.join(folge)`), `anzahl`, `start` und `farbe`. Mit
`continue` überspringst du Fälle mit zu wenigen Kandidaten. Vergleiche `beste_auswahl(...)[1]` mit `durchprobiert(...)`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import itertools

def finde_kandidaten(farben, farbe):
    return [j for j in range(len(farben)) if farben[j] == farbe]

def tauschkosten(auswahl, start):
    summe = 0
    for i in range(len(auswahl)):
        summe = summe + abs(auswahl[i] - (start + i))
    return summe

def beste_auswahl(farben, start, anzahl, farbe):
    kandidaten = finde_kandidaten(farben, farbe)
    beste = None
    beste_kosten = None
    for a in range(len(kandidaten) - anzahl + 1):
        auswahl = kandidaten[a:a + anzahl]
        kosten = tauschkosten(auswahl, start)
        if beste_kosten is None or kosten < beste_kosten:
            beste = auswahl
            beste_kosten = kosten
    return beste, beste_kosten

def durchprobiert(farben, start, anzahl, farbe):
    """Kleinste Kosten über ALLE Auswahlen von anzahl Ringeln dieser Farbe."""
    kandidaten = finde_kandidaten(farben, farbe)
    beste_kosten = None
    for auswahl in itertools.combinations(kandidaten, anzahl):
        kosten = tauschkosten(list(auswahl), start)
        if beste_kosten is None or kosten < beste_kosten:
            beste_kosten = kosten
    return beste_kosten

def vergleiche_fs(laenge):
    alle = 0
    gleich = 0
    naiv_falsch = 0
    for folge in itertools.product('gr', repeat=laenge):
        farben = ''.join(folge)
        for anzahl in range(1, 4):
            for start in range(0, laenge - anzahl + 1):
                for farbe in 'gr':
                    kandidaten = finde_kandidaten(farben, farbe)
                    if len(kandidaten) < anzahl:
                        continue
                    alle = alle + 1
                    genau = durchprobiert(farben, start, anzahl, farbe)
                    if beste_auswahl(farben, start, anzahl, farbe)[1] == genau:
                        gleich = gleich + 1
                    if tauschkosten(kandidaten[:anzahl], start) > genau:
                        naiv_falsch = naiv_falsch + 1
    return alle, gleich, naiv_falsch

alle, gleich, naiv_falsch = vergleiche_fs(8)
print(alle, 'Fälle,', gleich, 'mal Fenster = beste Auswahl,', naiv_falsch, 'mal ist "die ersten von links" schlechter')
```

In allen 10 166 Fällen findet das Fenster-Verfahren dieselben Kosten wie das Durchprobieren aller Auswahlen. Die naive Auswahl „die ersten von links“ ist dagegen in 5 786 Fällen
schlechter, denn die ersten Kandidaten von links liegen nicht unbedingt in der Nähe von `start`. Das ist **kein Beweis**: Der Test deckt nur Folgen der Länge 8 und `anzahl` bis 3 ab.
Er macht das Verfahren aber glaubwürdig.

</details>

#### Frage F.4

1. Der Test in F.3 bestätigt das Fenster-Verfahren für Folgen der Länge 8. Warum **beweist** das noch nicht, dass es immer stimmt? Was würde ein einziger Fall mit
   `gleich` kleiner als `alle` bedeuten?
2. Wie viele Fenster probiert `beste_auswahl` bei `c` Kandidaten und `anzahl` Ringeln, und wie viel Arbeit macht jedes Fenster? Wie viele Auswahlen probiert dagegen
   `durchprobiert`?
3. Das Verfahren probiert nur Fenster aus **nebeneinanderliegenden** Kandidaten. Welche Auswahlen lässt es damit aus, und warum erwartet man, dass das nichts schadet?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Der Test prüft nur Folgen der Länge 8 und `anzahl` bis 3. Für längere Folgen oder größere `anzahl` könnte es ein Gegenbeispiel geben, das wir nicht gesehen haben. Ein einziger
   Fall mit `gleich < alle` wäre dagegen ein **Gegenbeispiel** und würde das Verfahren widerlegen. Für den Beweis braucht man ein Austauschargument.
2. Es sind `c - anzahl + 1` Fenster, jedes kostet `anzahl` Additionen: **O(c · anzahl)**. `durchprobiert` probiert **c über anzahl** Auswahlen, bei 40 Kandidaten und `anzahl = 10` schon
   über 800 Millionen.
3. Es lässt Auswahlen aus, bei denen zwischen zwei gewählten Ringeln noch ein **ungewählter** gleichfarbiger liegt. Man vermutet, dass ein Austausch
   einer der beiden gewählten gegen den Ringel dazwischen nie teurer wird. Ein Beweis dafür wäre ein kleines Austauschargument (eine gute Knobelaufgabe); hier stützt nur der Test aus F.3 die Vermutung.

</details>

## Das hast du gelernt

```python
# Gierig: nach einer Regel sortieren, dann der Reihe nach entscheiden
reihenfolge = sorted(termine, key=lambda t: t[1])        # früh endend zuerst
gewaehlt = []
for termin in reihenfolge:
    if passt(termin, gewaehlt):                           # keine Überschneidung
        gewaehlt.append(termin)

# Prüfer für kleine Beispiele: alle Möglichkeiten durchprobieren
for auswahl in itertools.combinations(termine, k):
    ...

# Gegenbeispiele mit Zufall suchen (fester Anfang: random.seed(1))
for versuch in range(500):
    daten = zufalls_daten()
    if gierig(daten) != genau(daten):
        print(daten)
```

- **Gierig** heißt: Immer die scheinbar beste Entscheidung nehmen, nie zurückgehen. Das ist schnell (oft **O(n log n)** wegen des Sortierens), aber nicht immer richtig.
- **Reihenfolge ist Teil des Verfahrens**: größte Münze zuerst, früh endender Termin zuerst. Die falsche Reihenfolge (zum Beispiel „kürzester Termin zuerst“) gibt falsche Ergebnisse.
- **Gegenbeispiele finden**: das gierige Verfahren mit Durchprobieren oder einer Tabelle (Notebook 11) vergleichen, auch mit vielen Zufallseingaben. Ein Gegenbeispiel **widerlegt**,
  bestandene Tests **beweisen nichts**.
- **Begründen mit einem Austauschargument**: Nimm eine beste Lösung, ersetze ihren ersten Teil durch die gierige Wahl, und zeige, dass sie nicht schlechter wird. Dann
  wiederholt man das für den Rest.
- Oft hat eine Aufgabe **mehrere Ziele nacheinander** (erst wenige Stopps, dann billig): Die gierige Regel gilt dann nur für das Ziel, für das man sie begründet hat.
- Grenzfälle (Ende gleich Anfang, gleiche Schlüssel beim Sortieren) vor dem Abgeben testen.

### Aufgaben der 1. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Urlaubsfahrt](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/38_Urlaubsfahrt/urlaubsfahrt_loesung.ipynb) (38. BwInf): wenige Tankstopps gierig, Tankmenge mit Austausch, Vergleich mit Gedächtnis
- [Frühsport](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf1/31_Fruehsport/fruehsport_loesung.ipynb) (31.): gieriges Besorgen der Farben beim Tauschen

Im nächsten Notebook (13) geht es um die **binäre Suche**: Man **rät einen Zielwert** und prüft nur „Geht es mit höchstens `x`?“, ein gieriger Test genügt oft. Durch Halbieren des
Suchbereichs findet man den besten Wert in sehr wenigen Schritten.